# LeetCode 966 — Vowel Spellchecker bằng Python

Link đề: https://leetcode.com/problems/vowel-spellchecker/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Hash Table (bảng băm), String (chuỗi)

Skeleton LeetCode:

```python
class Solution:
    def spellchecker(self, wordlist: List[str], queries: List[str]) -> List[str]:
        ...
```

Ghi chú: với mỗi từ truy vấn (`query`) ta phải trả về từ "đúng" trong `wordlist` theo **ba mức ưu tiên**: (1) khớp **y hệt** (phân biệt hoa/thường); (2) khớp **không phân biệt hoa/thường** (lấy từ **đầu tiên** trong danh sách); (3) khớp sau khi **thay mỗi nguyên âm bằng một nguyên âm bất kỳ** (cũng lấy từ đầu tiên); nếu không có thì trả `""`. Bài này là bài **thiết kế khóa băm**: mỗi mức ưu tiên là một "khóa chuẩn hóa" khác nhau, và mọi cạm bẫy nằm ở **thứ tự ưu tiên** và **"lấy từ đầu tiên"**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ba loại khớp, thứ tự ưu tiên, quy tắc "đầu tiên trong `wordlist`".
2. Nắm các công cụ Python cần thiết: `str.lower()`, `set`, `dict`, `dict.setdefault`, `dict.get`, `itertools.product`, `reversed`, và dựng chuỗi bằng `''.join(...)`.
3. Hiểu nguyên lý **chuẩn hóa khóa**: hai từ "giống nhau ở mức X" khi chúng có cùng khóa chuẩn hóa ở mức X; với mức 3 khóa là chuỗi chữ thường mà mọi nguyên âm đổi thành ký tự đại diện `*`.
4. Đi qua 4 cách: quét tuyến tính mọi từ cho từng truy vấn (brute force), ba bảng băm với `setdefault` (cách nộp chính), một bảng băm dựng bằng cách duyệt ngược danh sách, và liệt kê mọi biến thể nguyên âm của truy vấn rồi lấy chỉ số nhỏ nhất.
5. Tránh các lỗi kinh điển: bảng băm ghi đè (lấy từ **cuối** thay vì **đầu**), kiểm tra sai thứ tự ưu tiên, quên `.lower()` trước khi che nguyên âm, bỏ qua bước khớp chính xác, và **xóa** nguyên âm thay vì thay bằng ký tự đại diện.

## 2. Đề bài & ví dụ

Cho `wordlist`; ta xây một bộ sửa lỗi chính tả biến mỗi từ truy vấn (`query`) thành một từ đúng. Bộ sửa lỗi xử lý hai loại lỗi:

- **Viết hoa:** nếu `query` khớp một từ trong `wordlist` **không phân biệt hoa/thường**, trả về từ đó **với kiểu hoa/thường như trong `wordlist`**.
- **Lỗi nguyên âm:** nếu sau khi thay các nguyên âm (`a`, `e`, `i`, `o`, `u`) của `query` bằng **bất kỳ nguyên âm nào** (từng nguyên âm độc lập) mà khớp một từ trong `wordlist` (không phân biệt hoa/thường), trả về từ đó với kiểu hoa/thường như trong `wordlist`.

Quy tắc ưu tiên:

1. `query` khớp **chính xác** (phân biệt hoa/thường) một từ trong `wordlist` thì trả lại chính từ đó.
2. Khớp sau khi bỏ qua hoa/thường: trả về từ **đầu tiên** như vậy trong `wordlist`.
3. Khớp sau khi bỏ qua lỗi nguyên âm: trả về từ **đầu tiên** như vậy.
4. Không khớp: trả về chuỗi rỗng `""`.

Với `queries`, trả về mảng `answer` mà `answer[i]` là đáp án cho `queries[i]`.

**Ví dụ 1:** `wordlist = ["KiTe","kite","hare","Hare"]`, `queries = ["kite","Kite","KiTe","Hare","HARE","Hear","hear","keti","keet","keto"]` -> `["kite","KiTe","KiTe","Hare","hare","","","KiTe","","KiTe"]`.

**Ví dụ 2:** `wordlist = ["yellow"]`, `queries = ["YellOw"]` -> `["yellow"]`.

**Ràng buộc:**

- `1 <= wordlist.length, queries.length <= 5000`
- `1 <= wordlist[i].length, queries[i].length <= 7`
- `wordlist[i]` và `queries[i]` chỉ gồm chữ cái tiếng Anh (hoa hoặc thường).

In [1]:
from typing import List, Dict, Optional
from itertools import product
import random, time

examples = [
    (["KiTe", "kite", "hare", "Hare"],
     ["kite", "Kite", "KiTe", "Hare", "HARE", "Hear", "hear", "keti", "keet", "keto"],
     ["kite", "KiTe", "KiTe", "Hare", "hare", "", "", "KiTe", "", "KiTe"]),
    (["yellow"], ["YellOw"], ["yellow"]),
]

for wl, qs, expected in examples:
    print(f"wordlist={wl}\nqueries={qs}\nkỳ vọng={expected}\n")

wordlist=['KiTe', 'kite', 'hare', 'Hare']
queries=['kite', 'Kite', 'KiTe', 'Hare', 'HARE', 'Hear', 'hear', 'keti', 'keet', 'keto']
kỳ vọng=['kite', 'KiTe', 'KiTe', 'Hare', 'hare', '', '', 'KiTe', '', 'KiTe']

wordlist=['yellow']
queries=['YellOw']
kỳ vọng=['yellow']



## 3. Kiến thức nền

### 3.1. `str.lower()` và so sánh không phân biệt hoa/thường

`"KiTe".lower()` cho `"kite"`. Hai từ khớp "không phân biệt hoa/thường" khi `a.lower() == b.lower()`. Vì chuỗi trong Python **bất biến**, `lower()` trả về chuỗi mới.

In [2]:
print("KiTe".lower(), "HARE".lower(), "YellOw".lower() == "yellow")
print("Hear".lower() == "hare", "chữ hoa/thường khác nhau:", "Kite" == "kite")

kite hare True
False chữ hoa/thường khác nhau: False


### 3.2. `set`, `dict`, `setdefault`

- `set` kiểm tra thuộc tập trong `O(1)` trung bình.
- `d.setdefault(k, v)` chỉ gán `d[k] = v` **nếu `k` chưa có**: đúng ngữ nghĩa "giữ phần tử đầu tiên".
- `d[k] = v` **ghi đè** mỗi lần: giữ phần tử **cuối cùng**. Đây là nguồn lỗi hay gặp.
- `d.get(k, "")` trả `""` khi thiếu khóa.

In [3]:
first, last = {}, {}
for w in ["KiTe", "kite", "KITE"]:
    first.setdefault(w.lower(), w)
    last[w.lower()] = w
print("setdefault giữ đầu:", first, "| gán thường giữ cuối:", last)
print("get thiếu khóa:", first.get("zzz", ""))

setdefault giữ đầu: {'kite': 'KiTe'} | gán thường giữ cuối: {'kite': 'KITE'}
get thiếu khóa: 


### 3.3. Khóa chuẩn hóa cho "lỗi nguyên âm"

Hai từ khớp "lỗi nguyên âm" ⇔ cùng độ dài và ở mọi vị trí: hoặc hai chữ cái (chữ thường) **bằng nhau**, hoặc **cả hai đều là nguyên âm**. Ta mã hóa điều đó bằng một khóa: chữ thường, rồi thay mọi nguyên âm bằng `*`. Hai từ có cùng khóa ⇔ chúng khớp mức 3.

In [4]:
VOWELS = set("aeiou")

def devowel(w: str) -> str:
    return "".join("*" if c in VOWELS else c for c in w.lower())

for w in ["Hare", "Hear", "hore", "yellow", "YOLLOW", "yeellow", "yllw", "AEIOU"]:
    print(f"{w:8} -> {devowel(w)}")

Hare     -> h*r*
Hear     -> h**r
hore     -> h*r*
yellow   -> y*ll*w
YOLLOW   -> y*ll*w
yeellow  -> y**ll*w
yllw     -> yllw
AEIOU    -> *****


Quan sát: `Hare` và `hore` cùng khóa `h*r*`; `yeellow` có khóa `y**ll*w` dài hơn `y*ll*w`, nên **không** khớp `yellow` (vì chỉ **thay**, không xóa nguyên âm).

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1 với `wordlist = ["KiTe","kite","hare","Hare"]`. Dựng ba bảng:

- Chính xác: `{KiTe, kite, hare, Hare}`.
- Không phân biệt hoa/thường (giữ từ đầu tiên): `kite -> KiTe`, `hare -> hare`.
- Lỗi nguyên âm (giữ từ đầu tiên): `k*t* -> KiTe`, `h*r* -> hare`.

Tra từng truy vấn:

- `Kite`: không chính xác; `kite` có trong bảng 2 -> `KiTe` (từ đầu tiên).
- `HARE`: không chính xác; `hare` có trong bảng 2 -> `hare` (đầu tiên, không phải `Hare`).
- `Hear`: `hear` không có trong bảng 2; khóa `h**r` không có trong bảng 3 (`h*r*` khác) -> `""`.
- `keti`: không có ở bảng 1 và 2; khóa `k*t*` ở bảng 3 -> `KiTe`.
- `keet`: khóa `k**t` không có -> `""`.

## 5. Cách 1 — Brute Force: quét tuyến tính cho từng truy vấn

**Ý tưởng.** Với mỗi `query`, quét `wordlist` ba lượt theo thứ tự ưu tiên, **đúng theo định nghĩa**:

1. Có từ `w == query` -> trả `query`.
2. Từ `w` đầu tiên có `w.lower() == query.lower()` -> trả `w`.
3. Từ `w` đầu tiên **cùng độ dài** và ở mọi vị trí hai ký tự (chữ thường) bằng nhau hoặc cùng là nguyên âm -> trả `w`.
4. Không có -> `""`.

Cách này không dùng bảng băm hay khóa chuẩn hóa nên cũng là **oracle độc lập** để kiểm các cách sau.

In [5]:
def spell_brute(wordlist: List[str], queries: List[str]) -> List[str]:
    res = []
    for q in queries:
        found = ""
        if any(w == q for w in wordlist):
            found = q
        else:
            for w in wordlist:
                if w.lower() == q.lower():
                    found = w
                    break
            else:
                for w in wordlist:
                    a, b = w.lower(), q.lower()
                    if len(a) == len(b) and all(x == y or (x in VOWELS and y in VOWELS) for x, y in zip(a, b)):
                        found = w
                        break
        res.append(found)
    return res

for wl, qs, expected in examples:
    got = spell_brute(wl, qs)
    print("OK" if got == expected else "SAI", got)
    assert got == expected

OK ['kite', 'KiTe', 'KiTe', 'Hare', 'hare', '', '', 'KiTe', '', 'KiTe']
OK ['yellow']


**Độ phức tạp và nút thắt.** `O(Q * W * L)` với `Q` truy vấn, `W` từ, `L <= 7` độ dài. Với `Q = W = 5000`: `25 * 10^6` lần so sánh từ, mỗi lần tạo chuỗi mới: quá chậm trong Python. Đo thử (chỉ ở cỡ nhỏ, rồi ngoại suy):

In [6]:
LET = "aeioukthrAEIOUKTHR"
rng = random.Random(966)

def rand_word(rng, alphabet=LET, lo=1, hi=7):
    return "".join(rng.choice(alphabet) for _ in range(rng.randint(lo, hi)))

for n in (100, 200, 400):
    wl = [rand_word(rng) for _ in range(n)]
    qs = [rand_word(rng) for _ in range(n)]
    t0 = time.perf_counter()
    spell_brute(wl, qs)
    dt = time.perf_counter() - t0
    print(f"W=Q={n:4}: {dt * 1000:8.1f} ms  -> ngoại suy W=Q=5000: ~{dt * (5000 / n) ** 2:8.1f} s")

W=Q= 100:      4.4 ms  -> ngoại suy W=Q=5000: ~    10.9 s
W=Q= 200:     13.8 ms  -> ngoại suy W=Q=5000: ~     8.6 s
W=Q= 400:     52.6 ms  -> ngoại suy W=Q=5000: ~     8.2 s


Nút thắt: mỗi truy vấn **quét lại toàn bộ** `wordlist` và tính lại `lower()` cho cả hai bên. Ta nên **chuẩn hóa `wordlist` một lần** thành khóa băm, rồi tra cứu `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Mỗi mức ưu tiên = một phép chuẩn hóa + một bảng băm.**

| Mức | Chuẩn hóa | Bảng | Giá trị |
|---|---|---|---|
| 1 | giữ nguyên | `set` các từ | chính từ truy vấn |
| 2 | `lower()` | `dict` khóa -> từ **đầu tiên** | từ trong `wordlist` |
| 3 | `lower()` rồi nguyên âm -> `*` | `dict` khóa -> từ **đầu tiên** | từ trong `wordlist` |

Xây bảng một lần `O(W * L)`, mỗi truy vấn kiểm tra tuần tự mức 1 -> 2 -> 3 trong `O(L)`: tổng `O((W + Q) * L)`.

Hai ý then chốt:

1. **"Đầu tiên"**: dùng `setdefault` (hoặc duyệt ngược rồi gán thường).
2. **Thứ tự kiểm tra** đúng ưu tiên: mức 1, rồi 2, rồi 3.

| Cách | Ý tưởng |
|---|---|
| 2 | Ba bảng với `setdefault` (nộp) |
| 3 | Duyệt ngược `wordlist` và gán thường (từ đầu tiên ghi cuối cùng nên thắng) |
| 4 | Liệt kê mọi biến thể nguyên âm của truy vấn, lấy chỉ số nhỏ nhất |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Ba bảng băm với `setdefault` (cách nộp chính)

**Giải thích từng đoạn.**

- `exact = set(wordlist)`: để kiểm tra khớp chính xác.
- `cap`: `w.lower() -> w`, chỉ giữ lần xuất hiện đầu tiên (`setdefault`).
- `vow`: `devowel(w) -> w`, cũng giữ lần đầu.
- Mỗi truy vấn: nếu `q in exact` thì trả `q`; nếu không và `q.lower() in cap` thì trả `cap[q.lower()]`; nếu không thì `vow.get(devowel(q), "")`.

In [7]:
def spell_maps(wordlist: List[str], queries: List[str]) -> List[str]:
    exact = set(wordlist)
    cap: Dict[str, str] = {}
    vow: Dict[str, str] = {}
    for w in wordlist:
        cap.setdefault(w.lower(), w)
        vow.setdefault(devowel(w), w)
    res = []
    for q in queries:
        if q in exact:
            res.append(q)
        elif q.lower() in cap:
            res.append(cap[q.lower()])
        else:
            res.append(vow.get(devowel(q), ""))
    return res

for wl, qs, expected in examples:
    got = spell_maps(wl, qs)
    print("OK" if got == expected else "SAI", got)
    assert got == expected

OK ['kite', 'KiTe', 'KiTe', 'Hare', 'hare', '', '', 'KiTe', '', 'KiTe']
OK ['yellow']


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def spellchecker(self, wordlist: List[str], queries: List[str]) -> List[str]:
        def mask(s: str) -> str:
            return "".join("*" if c in "aeiou" else c for c in s.lower())

        exact = set(wordlist)
        cap, vow = {}, {}
        for w in wordlist:
            cap.setdefault(w.lower(), w)
            vow.setdefault(mask(w), w)

        ans = []
        for q in queries:
            if q in exact:
                ans.append(q)
            elif q.lower() in cap:
                ans.append(cap[q.lower()])
            else:
                ans.append(vow.get(mask(q), ""))
        return ans

sol_maps = Solution().spellchecker
for wl, qs, expected in examples:
    assert sol_maps(wl, qs) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O((W + Q) * L)`, bộ nhớ `O(W * L)`.

---

### 7.2. Cách 3 — Duyệt ngược `wordlist` và gán thường

**Ý tưởng.** Nếu duyệt `wordlist` **từ cuối về đầu** và gán `d[k] = w` bình thường, lần gán **cuối cùng** là từ **đầu tiên** của danh sách. Không cần `setdefault`. Ta còn gộp hai bảng vào **một** `dict` bằng cách gắn thêm **tiền tố mức** vào khóa (`("c", ...)`, `("v", ...)`).

**Giải thích từng đoạn.**

- Khóa mức 2: `(2, w.lower())`; khóa mức 3: `(3, devowel(w))`; hai loại không bao giờ va chạm nhờ tiền tố.
- Mức 1 vẫn là `set`.

In [9]:
def spell_reversed(wordlist: List[str], queries: List[str]) -> List[str]:
    exact = set(wordlist)
    table: Dict[tuple, str] = {}
    for w in reversed(wordlist):
        table[(2, w.lower())] = w
        table[(3, devowel(w))] = w
    res = []
    for q in queries:
        if q in exact:
            res.append(q)
        elif (2, q.lower()) in table:
            res.append(table[(2, q.lower())])
        else:
            res.append(table.get((3, devowel(q)), ""))
    return res

for wl, qs, expected in examples:
    got = spell_reversed(wl, qs)
    print("OK" if got == expected else "SAI", got)
    assert got == expected

OK ['kite', 'KiTe', 'KiTe', 'Hare', 'hare', '', '', 'KiTe', '', 'KiTe']
OK ['yellow']


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def spellchecker(self, wordlist: List[str], queries: List[str]) -> List[str]:
        def mask(s: str) -> str:
            return "".join("*" if c in "aeiou" else c for c in s.lower())

        exact = set(wordlist)
        table = {}
        for w in reversed(wordlist):          # duyệt ngược: từ đầu tiên ghi sau cùng nên thắng
            table[(2, w.lower())] = w
            table[(3, mask(w))] = w
        ans = []
        for q in queries:
            if q in exact:
                ans.append(q)
            else:
                ans.append(table.get((2, q.lower())) or table.get((3, mask(q)), ""))
        return ans

sol_reversed = Solution().spellchecker
for wl, qs, expected in examples:
    assert sol_reversed(wl, qs) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


Lưu ý: `table.get((2, ...)) or ...` an toàn vì mọi từ trong `wordlist` đều **không rỗng** (độ dài `>= 1`), nên giá trị tìm được luôn "truthy".

**Độ phức tạp (Cách 3):** như Cách 2: `O((W + Q) * L)` thời gian, `O(W * L)` bộ nhớ.

---

### 7.3. Cách 4 — Liệt kê mọi biến thể nguyên âm của truy vấn

**Ý tưởng khác.** Thay vì "che" nguyên âm của `wordlist`, ta **sinh mọi biến thể** của truy vấn: mỗi nguyên âm của `q.lower()` thay bằng một trong 5 nguyên âm (`itertools.product`). Với mỗi biến thể có trong `wordlist` (không phân biệt hoa/thường) ta lấy **chỉ số đầu tiên**; đáp án là từ có **chỉ số nhỏ nhất**. Cách này bám sát câu chữ của đề ("thay bằng nguyên âm bất kỳ") và độc lập với khóa `*`.

**Giải thích từng đoạn.**

- `first_idx[lowercase] = chỉ số đầu tiên` của mỗi từ (chữ thường).
- Với truy vấn có `v` nguyên âm có `5^v` biến thể (`v <= 7`: tối đa `78125`): chậm hơn Cách 2 nhưng vẫn đúng.

In [11]:
def spell_variants(wordlist: List[str], queries: List[str]) -> List[str]:
    exact = set(wordlist)
    first_idx: Dict[str, int] = {}
    for i, w in enumerate(wordlist):
        first_idx.setdefault(w.lower(), i)
    res = []
    for q in queries:
        if q in exact:
            res.append(q)
            continue
        low = q.lower()
        if low in first_idx:
            res.append(wordlist[first_idx[low]])
            continue
        pos = [i for i, c in enumerate(low) if c in VOWELS]
        best = None
        for combo in product("aeiou", repeat=len(pos)):
            chars = list(low)
            for p, c in zip(pos, combo):
                chars[p] = c
            idx = first_idx.get("".join(chars))
            if idx is not None and (best is None or idx < best):
                best = idx
        res.append(wordlist[best] if best is not None else "")
    return res

for wl, qs, expected in examples:
    got = spell_variants(wl, qs)
    print("OK" if got == expected else "SAI", got)
    assert got == expected

OK ['kite', 'KiTe', 'KiTe', 'Hare', 'hare', '', '', 'KiTe', '', 'KiTe']
OK ['yellow']


**Code nộp LeetCode (Cách 4):** (đúng nhưng có thể chậm nếu nhiều truy vấn có nhiều nguyên âm; chỉ để so sánh)

In [12]:
class Solution:
    def spellchecker(self, wordlist: List[str], queries: List[str]) -> List[str]:
        from itertools import product
        exact = set(wordlist)
        first = {}
        for i, w in enumerate(wordlist):
            first.setdefault(w.lower(), i)
        ans = []
        for q in queries:
            if q in exact:
                ans.append(q)
                continue
            low = q.lower()
            if low in first:
                ans.append(wordlist[first[low]])
                continue
            pos = [i for i, c in enumerate(low) if c in "aeiou"]
            best = None
            for combo in product("aeiou", repeat=len(pos)):
                chars = list(low)
                for p, c in zip(pos, combo):
                    chars[p] = c
                idx = first.get("".join(chars))
                if idx is not None and (best is None or idx < best):
                    best = idx
            ans.append(wordlist[best] if best is not None else "")
        return ans

sol_variants = Solution().spellchecker
for wl, qs, expected in examples:
    assert sol_variants(wl, qs) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(W * L + Q * 5^v * L)` với `v <= 7` số nguyên âm của truy vấn; `O(W * L)` bộ nhớ.

## 8. TRACE từng bước

### 8.1. Dựng ba bảng cho ví dụ 1

Theo dõi từng từ của `wordlist = ["KiTe","kite","hare","Hare"]`: bảng nào được gán, bảng nào bị `setdefault` bỏ qua.

In [13]:
def trace_build(wordlist):
    exact = set()
    cap, vow = {}, {}
    for i, w in enumerate(wordlist):
        exact.add(w)
        k2, k3 = w.lower(), devowel(w)
        a = "gán" if k2 not in cap else f"bỏ qua (đã có {cap[k2]!r})"
        b = "gán" if k3 not in vow else f"bỏ qua (đã có {vow[k3]!r})"
        cap.setdefault(k2, w)
        vow.setdefault(k3, w)
        print(f"i={i} w={w!r:7} cap[{k2!r}]: {a:22} vow[{k3!r}]: {b}")
    print("exact =", sorted(exact))
    print("cap   =", cap)
    print("vow   =", vow)
    return exact, cap, vow

exact, cap, vow = trace_build(examples[0][0])
assert cap == {"kite": "KiTe", "hare": "hare"} and vow == {"k*t*": "KiTe", "h*r*": "hare"}

i=0 w='KiTe'  cap['kite']: gán                    vow['k*t*']: gán
i=1 w='kite'  cap['kite']: bỏ qua (đã có 'KiTe')  vow['k*t*']: bỏ qua (đã có 'KiTe')
i=2 w='hare'  cap['hare']: gán                    vow['h*r*']: gán
i=3 w='Hare'  cap['hare']: bỏ qua (đã có 'hare')  vow['h*r*']: bỏ qua (đã có 'hare')
exact = ['Hare', 'KiTe', 'hare', 'kite']
cap   = {'kite': 'KiTe', 'hare': 'hare'}
vow   = {'k*t*': 'KiTe', 'h*r*': 'hare'}


### 8.2. Tra từng truy vấn của ví dụ 1

Với mỗi truy vấn in ra mức nào quyết định kết quả.

In [14]:
def trace_query(exact, cap, vow, qs):
    out = []
    for q in qs:
        if q in exact:
            how, ans = "mức 1 (chính xác)", q
        elif q.lower() in cap:
            how, ans = "mức 2 (hoa/thường)", cap[q.lower()]
        elif devowel(q) in vow:
            how, ans = "mức 3 (nguyên âm)", vow[devowel(q)]
        else:
            how, ans = "không khớp", ""
        print(f"{q:5} -> {ans!r:7} {how}")
        out.append(ans)
    return out

got = trace_query(exact, cap, vow, examples[0][1])
assert got == examples[0][2]

kite  -> 'kite'  mức 1 (chính xác)
Kite  -> 'KiTe'  mức 2 (hoa/thường)
KiTe  -> 'KiTe'  mức 1 (chính xác)
Hare  -> 'Hare'  mức 1 (chính xác)
HARE  -> 'hare'  mức 2 (hoa/thường)
Hear  -> ''      không khớp
hear  -> ''      không khớp
keti  -> 'KiTe'  mức 3 (nguyên âm)
keet  -> ''      không khớp
keto  -> 'KiTe'  mức 3 (nguyên âm)


### 8.3. Ưu tiên giữa mức 2 và mức 3

`wordlist = ["hore", "Hare"]`, truy vấn `hare`. Mức 3 sẽ khớp `hore` (đứng trước), nhưng mức 2 khớp `Hare` nên **mức 2 thắng** dù nó đứng sau.

In [15]:
wl, qs = ["hore", "Hare"], ["hare"]
print("đáp án đúng:", spell_maps(wl, qs))
assert spell_maps(wl, qs) == ["Hare"] and spell_brute(wl, qs) == ["Hare"]

đáp án đúng: ['Hare']


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (ba bảng băm với `setdefault`): ngắn, rõ ràng và `O((W + Q) * L)`.

```python
class Solution:
    def spellchecker(self, wordlist: List[str], queries: List[str]) -> List[str]:
        def mask(s: str) -> str:
            return "".join("*" if c in "aeiou" else c for c in s.lower())

        exact = set(wordlist)
        cap, vow = {}, {}
        for w in wordlist:
            cap.setdefault(w.lower(), w)
            vow.setdefault(mask(w), w)

        ans = []
        for q in queries:
            if q in exact:
                ans.append(q)
            elif q.lower() in cap:
                ans.append(cap[q.lower()])
            else:
                ans.append(vow.get(mask(q), ""))
        return ans
```

In [16]:
class Solution:
    def spellchecker(self, wordlist: List[str], queries: List[str]) -> List[str]:
        def mask(s: str) -> str:
            return "".join("*" if c in "aeiou" else c for c in s.lower())

        exact = set(wordlist)
        cap, vow = {}, {}
        for w in wordlist:
            cap.setdefault(w.lower(), w)
            vow.setdefault(mask(w), w)

        ans = []
        for q in queries:
            if q in exact:
                ans.append(q)
            elif q.lower() in cap:
                ans.append(cap[q.lower()])
            else:
                ans.append(vow.get(mask(q), ""))
        return ans

sol = Solution()
for wl, qs, expected in examples:
    got = sol.spellchecker(wl, qs)
    print("OK" if got == expected else "SAI", got)
    assert got == expected

OK ['kite', 'KiTe', 'KiTe', 'Hare', 'hare', '', '', 'KiTe', '', 'KiTe']
OK ['yellow']


## 10. Vì sao đúng

**(a) Chuẩn hóa mức 2.** Hai từ khớp "không phân biệt hoa/thường" ⇔ `a.lower() == b.lower()`. Bảng `cap` ánh xạ mỗi chuỗi thường tới từ **đầu tiên** của `wordlist` có chuỗi thường đó (`setdefault` không ghi đè). Do đó `cap[q.lower()]` là từ đầu tiên khớp mức 2.

**(b) Chuẩn hóa mức 3.** Từ `w` khớp `q` ở mức 3 ⇔ tồn tại cách thay mỗi nguyên âm của `q` bằng một nguyên âm sao cho `lower` của kết quả bằng `w.lower()`. Điều đó xảy ra ⇔ cùng độ dài, và tại mỗi vị trí hoặc hai ký tự (đã thường) bằng nhau (khi là phụ âm, hai ký tự phải bằng nhau; khi là nguyên âm ở `q`, ký tự ở `w` phải là nguyên âm nào đó) ⇔ `mask(q) == mask(w)`. Chiều ngược lại: nếu `mask` bằng nhau thì tại các vị trí `*` cả hai là nguyên âm, nên thay nguyên âm của `q` bằng nguyên âm tương ứng của `w` cho ra đúng `w.lower()`. Lưu ý: nếu `w` có phụ âm ở vị trí mà `q` có nguyên âm thì `mask` khác nhau (`*` so với phụ âm): đúng, vì ta chỉ được thay nguyên âm bằng **nguyên âm**.

**(c) Thứ tự ưu tiên.** Chương trình kiểm tra mức 1, rồi 2, rồi 3, đúng thứ tự đề bài; mức thấp hơn chỉ được xét khi mức cao hơn thất bại.

**(d) "Đầu tiên".** `setdefault` (hay duyệt ngược) bảo đảm từ trả về là từ có chỉ số nhỏ nhất trong số các từ cùng khóa. Mức 1 trả về `q` (chính nó) nên không cần quan tâm thứ tự.

**(e) Độ phức tạp.** Dựng bảng `O(W * L)`, mỗi truy vấn `O(L)`.

**Kiểm chứng bằng máy.** Oracle độc lập (Cách 1) bám sát định nghĩa từng ký tự, không dùng khóa băm; Cách 4 sinh biến thể theo câu chữ của đề. So sánh cả bốn cách và lời giải nộp trên rất nhiều test ngẫu nhiên có seed (bảng chữ cái nhỏ để tạo nhiều va chạm).

In [17]:
def make_case(rng, nw, nq, alphabet, lo=1, hi=7):
    wl = ["".join(rng.choice(alphabet) for _ in range(rng.randint(lo, hi))) for _ in range(nw)]
    qs = []
    for _ in range(nq):
        if wl and rng.random() < 0.6:                       # truy vấn "gần" một từ có sẵn: đổi hoa/thường hoặc nguyên âm
            base = rng.choice(wl)
            chars = []
            for c in base:
                r = rng.random()
                if r < 0.3:
                    c = c.swapcase()
                elif r < 0.55 and c.lower() in VOWELS:
                    c = rng.choice("aeiouAEIOU")
                chars.append(c)
            qs.append("".join(chars))
        else:
            qs.append("".join(rng.choice(alphabet) for _ in range(rng.randint(lo, hi))))
    return wl, qs

rng = random.Random(1966)
cnt = {"exact": 0, "cap": 0, "vow": 0, "none": 0}
for _ in range(400):
    wl, qs = make_case(rng, rng.randint(1, 8), rng.randint(1, 8), rng.choice(["aAeEkK", "aeiouAEIOUk", "abAB", "aeiouAEIOUbcdBCD"]), 1, 5)
    ref = spell_brute(wl, qs)
    assert spell_maps(wl, qs) == ref == spell_reversed(wl, qs) == spell_variants(wl, qs) == sol.spellchecker(wl, qs), (wl, qs)
    for q, r in zip(qs, ref):
        cnt["exact" if r == q else "none" if r == "" else "cap" if r.lower() == q.lower() else "vow"] += 1
print("400 ca ngẫu nhiên khớp oracle; phân bố kết quả:", cnt)
assert all(v > 0 for v in cnt.values())

400 ca ngẫu nhiên khớp oracle; phân bố kết quả: {'exact': 293, 'cap': 473, 'vow': 451, 'none': 558}


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Quét tuyến tính | `O(Q * W * L)` | `O(1)` |
| 2. Ba bảng `setdefault` | `O((W + Q) * L)` | `O(W * L)` |
| 3. Duyệt ngược + một bảng | `O((W + Q) * L)` | `O(W * L)` |
| 4. Liệt kê biến thể | `O(W * L + Q * 5^v * L)` | `O(W * L)` |

Đo ở cỡ tối đa (`W = Q = 5000`); Cách 1 chỉ đo ở cỡ nhỏ ở trên. Cách 4 đo thêm trường hợp xấu (mọi truy vấn toàn nguyên âm).

In [18]:
rng = random.Random(11)
wl_big, qs_big = make_case(rng, 5000, 5000, "aeiouAEIOUbcdfgBCDFG")
ref = spell_maps(wl_big, qs_big)
for name, fn in (("Cách 2", spell_maps), ("Cách 3", spell_reversed), ("nộp", sol.spellchecker), ("Cách 4", spell_variants)):
    t0 = time.perf_counter()
    got = fn(wl_big, qs_big)
    dt = (time.perf_counter() - t0) * 1000
    assert got == ref, name
    print(f"{name:7} (W=Q=5000, trường hợp thường): {dt:9.1f} ms")

# Cách 4 ở trường hợp xấu: truy vấn dài 7 ký tự toàn nguyên âm -> 5^7 = 78125 biến thể mỗi truy vấn
worst_q = ["".join(rng.choice("aeiou") for _ in range(7)) + "" for _ in range(8)]
worst_wl = ["bcdfghk"] * 3
t0 = time.perf_counter()
spell_variants(worst_wl, worst_q)
print(f"Cách 4 xấu nhất: chỉ 8 truy vấn toàn nguyên âm: {(time.perf_counter() - t0) * 1000:9.1f} ms (5000 truy vấn sẽ ~{(time.perf_counter() - t0) * 625:6.0f} s)")
t0 = time.perf_counter()
sol.spellchecker(worst_wl, worst_q * 625)
print(f"Cách nộp, 5000 truy vấn xấu tương tự: {(time.perf_counter() - t0) * 1000:9.1f} ms")

Cách 2  (W=Q=5000, trường hợp thường):      10.8 ms
Cách 3  (W=Q=5000, trường hợp thường):      13.5 ms
nộp     (W=Q=5000, trường hợp thường):      11.0 ms


Cách 4  (W=Q=5000, trường hợp thường):    1429.4 ms


Cách 4 xấu nhất: chỉ 8 truy vấn toàn nguyên âm:     426.0 ms (5000 truy vấn sẽ ~   266 s)
Cách nộp, 5000 truy vấn xấu tương tự:       4.0 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), các phản ví dụ cho cách sai (tìm bằng tìm kiếm có seed), test ngẫu nhiên đối chiếu oracle, tính chất, và test lớn ở đúng giới hạn.

In [19]:
def all_solutions():
    return {
        "brute": spell_brute, "maps": spell_maps, "reversed": spell_reversed, "variants": spell_variants,
        "sol_maps": sol_maps, "sol_reversed": sol_reversed, "sol_variants": sol_variants, "final": Solution().spellchecker,
    }

def check_all(wl, qs, expected=None, skip_variants=False):
    ref = spell_brute(wl, qs)
    for name, fn in all_solutions().items():
        if skip_variants and "variants" in name:
            continue
        assert fn(list(wl), list(qs)) == ref, (name, wl, qs, fn(list(wl), list(qs)), ref)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", wl, qs, expected, ref)
    return ref

# 12.1 ví dụ đề
for wl, qs, expected in examples:
    check_all(wl, qs, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (["a"], ["A", "e", "b"], ["a", "a", ""]),                       # nguyên âm đơn: hoa/thường, đổi nguyên âm, phụ âm không khớp
    (["Ab"], ["aB", "AB", "Ab", "ab"], ["Ab", "Ab", "Ab", "Ab"]),   # mọi biến thể hoa/thường trả về dạng trong wordlist
    (["Aa"], ["eE", "Ae", "ea", "xx"], ["Aa", "Aa", "Aa", ""]),     # chỉ khác nguyên âm
    (["ab", "ab"], ["ab", "AB"], ["ab", "ab"]),                     # từ trùng lặp
    (["ae"], ["a", "aee", "ae"], ["", "", "ae"]),                   # khác độ dài không khớp
    (["hare"], ["hore", "jore", "HORE"], ["hare", "", "hare"]),     # nguyên âm khớp, phụ âm khác thì không; HORE cần lower trước
    (["Yellow"], ["yollow", "yeellow", "yllw", "YELLOW"], ["Yellow", "", "", "Yellow"]),  # ví dụ trong đề (xóa nguyên âm không hợp lệ)
    (["hore", "Hare"], ["hare"], ["Hare"]),                         # mức 2 thắng mức 3 dù đứng sau
    (["kite", "KiTe"], ["KiTe", "kite", "KITE"], ["KiTe", "kite", "kite"]),  # chính xác thắng; KITE lấy từ đầu tiên
    (["bcd"], ["bcd", "BCD", "bad"], ["bcd", "bcd", ""]),           # không có nguyên âm
    (["AEIOU"], ["uoiea", "aeiou", "AEIOUa"], ["AEIOU", "AEIOU", ""]),
    (["Xe", "xI"], ["XA", "xo"], ["Xe", "Xe"]),                     # nhiều từ cùng khóa mức 3: lấy từ đầu
]
for wl, qs, expected in edge:
    check_all(wl, qs, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 12 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_last_wins(wordlist, queries):            # gán thường (ghi đè) -> lấy từ CUỐI
    exact = set(wordlist)
    cap, vow = {}, {}
    for w in wordlist:
        cap[w.lower()] = w
        vow[devowel(w)] = w
    return [q if q in exact else cap[q.lower()] if q.lower() in cap else vow.get(devowel(q), "") for q in queries]

def wrong_order(wordlist, queries):                # kiểm tra mức 3 trước mức 2
    exact = set(wordlist)
    cap, vow = {}, {}
    for w in wordlist:
        cap.setdefault(w.lower(), w)
        vow.setdefault(devowel(w), w)
    out = []
    for q in queries:
        if q in exact:
            out.append(q)
        elif devowel(q) in vow:
            out.append(vow[devowel(q)])
        else:
            out.append(cap.get(q.lower(), ""))
    return out

def mask_no_lower(s):
    return "".join("*" if c in VOWELS else c for c in s)           # quên .lower()

def wrong_no_lower(wordlist, queries):             # che nguyên âm mà quên lower: 'O' không bị che
    exact = set(wordlist)
    cap, vow = {}, {}
    for w in wordlist:
        cap.setdefault(w.lower(), w)
        vow.setdefault(mask_no_lower(w), w)
    return [q if q in exact else cap[q.lower()] if q.lower() in cap else vow.get(mask_no_lower(q), "") for q in queries]

def wrong_no_exact(wordlist, queries):             # bỏ bước khớp chính xác
    cap, vow = {}, {}
    for w in wordlist:
        cap.setdefault(w.lower(), w)
        vow.setdefault(devowel(w), w)
    return [cap[q.lower()] if q.lower() in cap else vow.get(devowel(q), "") for q in queries]

def devowel_delete(w):
    return "".join(c for c in w.lower() if c not in VOWELS)

def wrong_delete(wordlist, queries):               # XÓA nguyên âm thay vì thay bằng '*'
    exact = set(wordlist)
    cap, vow = {}, {}
    for w in wordlist:
        cap.setdefault(w.lower(), w)
        vow.setdefault(devowel_delete(w), w)
    return [q if q in exact else cap[q.lower()] if q.lower() in cap else vow.get(devowel_delete(q), "") for q in queries]

def find_counter(wrong, seed, tries=60000):
    rng = random.Random(seed)
    for _ in range(tries):
        wl, qs = make_case(rng, rng.randint(1, 3), rng.randint(1, 2), "aeAEkK", 1, 3)
        if wrong(wl, qs) != spell_brute(wl, qs):
            return wl, qs
    return None

counters = {}
for name, fn in (("ghi đè (lấy từ cuối)", wrong_last_wins), ("sai thứ tự mức 2/mức 3", wrong_order),
                 ("quên lower trước khi che", wrong_no_lower), ("bỏ khớp chính xác", wrong_no_exact),
                 ("xóa nguyên âm thay vì *", wrong_delete)):
    case = find_counter(fn, seed=9)
    assert case is not None, name
    counters[name] = case
    wl, qs = case
    print(f"{name:26} wordlist={wl} queries={qs}\n{'':26} sai -> {fn(wl, qs)} | đúng -> {spell_brute(wl, qs)}")
    check_all(wl, qs)
print("12.3 phản ví dụ: OK")

ghi đè (lấy từ cuối)       wordlist=['E', 'kaE', 'a'] queries=['o']
                           sai -> ['a'] | đúng -> ['E']
sai thứ tự mức 2/mức 3     wordlist=['eA', 'A', 'eE'] queries=['EE', 'a']
                           sai -> ['eA', 'A'] | đúng -> ['eE', 'A']
quên lower trước khi che   wordlist=['ee', 'aAk'] queries=['Ue', 'K']
                           sai -> ['', ''] | đúng -> ['ee', '']
bỏ khớp chính xác          wordlist=['e', 'E', 'KK'] queries=['E', 'e']
                           sai -> ['e', 'e'] | đúng -> ['E', 'e']
xóa nguyên âm thay vì *    wordlist=['ee', 'aAk'] queries=['Ue', 'K']
                           sai -> ['ee', 'aAk'] | đúng -> ['ee', '']
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên: nhiều bảng chữ cái, kích thước nhỏ-vừa (so với oracle brute)
rng = random.Random(4966)
alphabets = ["aA", "aeiou", "aeiouAEIOU", "aAbB", "aeAEbB", "aeiouAEIOUbcdBCD", "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"]
for _ in range(600):
    wl, qs = make_case(rng, rng.randint(1, 30), rng.randint(1, 30), rng.choice(alphabets), 1, rng.randint(1, 7))
    check_all(wl, qs)
print("12.4 ngẫu nhiên (600 ca): OK")

# 12.5 test lớn đúng giới hạn: W = Q = 5000, độ dài 7; so các cách hiệu quả với nhau và với brute trên một mẫu truy vấn
rng = random.Random(5966)
for alpha in ("aeiouAEIOUbcBC", "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"):
    wl = ["".join(rng.choice(alpha) for _ in range(rng.randint(1, 7))) for _ in range(5000)]
    qs, _ = make_case(rng, 1, 5000, alpha)[1], None
    a = spell_maps(wl, qs)
    assert a == spell_reversed(wl, qs) == sol.spellchecker(wl, qs)
    sample = qs[:150]
    assert spell_brute(wl, sample) == a[:150]
print("12.5 test lớn W = Q = 5000: OK")

# 12.6 tính chất
rng = random.Random(6966)
for _ in range(300):
    wl, qs = make_case(rng, rng.randint(1, 15), rng.randint(1, 15), "aeiouAEIOUkt", 1, 5)
    res = sol.spellchecker(wl, qs)
    assert len(res) == len(qs)
    for q, r in zip(qs, res):
        if q in wl:
            assert r == q                                       # chính xác: trả lại chính nó
        if r:
            assert r in wl and len(r) == len(q)                 # đáp án là từ trong wordlist, cùng độ dài
    # đảo thứ tự truy vấn đảo thứ tự kết quả (mỗi truy vấn độc lập)
    assert sol.spellchecker(wl, qs[::-1]) == res[::-1]
    # thêm từ vào CUỐI wordlist không đổi kết quả của truy vấn đã có đáp án
    extra = sol.spellchecker(wl + ["zzzzzzz"], qs)
    assert all(r == e for r, e in zip(res, extra) if r)
print("12.6 tính chất: OK")

12.4 ngẫu nhiên (600 ca): OK
12.5 test lớn W = Q = 5000: OK


12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Gán thường vào `dict` (lấy từ cuối thay vì từ đầu)

`cap[w.lower()] = w` ghi đè mỗi lần gặp lại khóa nên giữ **từ cuối cùng**. Đề yêu cầu từ **đầu tiên**.

In [22]:
wl, qs = counters["ghi đè (lấy từ cuối)"]
print("wordlist =", wl, "| queries =", qs)
print("sai ->", wrong_last_wins(wl, qs), "| đúng ->", Solution().spellchecker(wl, qs))
assert wrong_last_wins(wl, qs) != Solution().spellchecker(wl, qs)

wordlist = ['E', 'kaE', 'a'] | queries = ['o']
sai -> ['a'] | đúng -> ['E']


Vì sao sai: hai từ cùng chuỗi thường (hoặc cùng khóa nguyên âm), từ đứng sau đè từ đứng trước. **Đúng:** `setdefault`, hoặc duyệt ngược rồi gán thường.

### Lỗi 2 — Kiểm tra mức 3 trước mức 2

Khớp lỗi nguyên âm có thể nhặt được từ đứng **trước** nhưng đề ưu tiên khớp hoa/thường.

In [23]:
wl, qs = counters["sai thứ tự mức 2/mức 3"]
print("wordlist =", wl, "| queries =", qs)
print("sai ->", wrong_order(wl, qs), "| đúng ->", Solution().spellchecker(wl, qs))
assert wrong_order(wl, qs) != Solution().spellchecker(wl, qs)
print("ví dụ đã phân tích ở mục 8.3:", wrong_order(["hore", "Hare"], ["hare"]), "(sai) so với", Solution().spellchecker(["hore", "Hare"], ["hare"]))

wordlist = ['eA', 'A', 'eE'] | queries = ['EE', 'a']
sai -> ['eA', 'A'] | đúng -> ['eE', 'A']
ví dụ đã phân tích ở mục 8.3: ['hore'] (sai) so với ['Hare']


Vì sao sai: mức ưu tiên là **chính xác > hoa/thường > nguyên âm**, không phải theo thứ tự trong `wordlist`. **Đúng:** kiểm tra theo đúng thứ tự ưu tiên.

### Lỗi 3 — Quên `.lower()` trước khi che nguyên âm

Nếu chỉ thay `a e i o u` (chữ thường) mà không hạ chữ hoa trước, nguyên âm in hoa như `O`, `E` không bị che.

In [24]:
wl, qs = counters["quên lower trước khi che"]
print("wordlist =", wl, "| queries =", qs)
print("sai ->", wrong_no_lower(wl, qs), "| đúng ->", Solution().spellchecker(wl, qs))
assert wrong_no_lower(wl, qs) != Solution().spellchecker(wl, qs)

wordlist = ['ee', 'aAk'] | queries = ['Ue', 'K']
sai -> ['', ''] | đúng -> ['ee', '']


Vì sao sai: khóa của `"HORE"` là `"HORE"` (không có `*`), khác khóa `"h*r*"` của `"hare"`. **Đúng:** `s.lower()` **trước** khi che.

### Lỗi 4 — Bỏ bước khớp chính xác

Khớp chính xác (phân biệt hoa/thường) phải được ưu tiên: nếu `query` có đúng trong `wordlist` thì trả lại **chính nó**, ngay cả khi có từ khác (cùng chữ thường) đứng trước.

In [25]:
wl, qs = counters["bỏ khớp chính xác"]
print("wordlist =", wl, "| queries =", qs)
print("sai ->", wrong_no_exact(wl, qs), "| đúng ->", Solution().spellchecker(wl, qs))
assert wrong_no_exact(wl, qs) != Solution().spellchecker(wl, qs)

wordlist = ['e', 'E', 'KK'] | queries = ['E', 'e']
sai -> ['e', 'e'] | đúng -> ['E', 'e']


Vì sao sai: không có bước 1 thì `"KiTe"` rơi vào mức 2 và nhận từ đầu tiên (`"kite"`). **Đúng:** `if q in exact: return q` trước tiên.

### Lỗi 5 — Xóa nguyên âm thay vì thay bằng ký tự đại diện

Đề cho phép **thay** nguyên âm bằng nguyên âm khác, **không** cho xóa hay chèn. Nếu xóa hết nguyên âm thì `"yeellow"` bị coi là khớp `"yellow"` (độ dài khác nhau).

In [26]:
wl, qs = counters["xóa nguyên âm thay vì *"]
print("wordlist =", wl, "| queries =", qs)
print("sai ->", wrong_delete(wl, qs), "| đúng ->", Solution().spellchecker(wl, qs))
assert wrong_delete(wl, qs) != Solution().spellchecker(wl, qs)
print("ví dụ của đề:", wrong_delete(["YellOw"], ["yeellow", "yllw"]), "(sai) so với", Solution().spellchecker(["YellOw"], ["yeellow", "yllw"]))
assert Solution().spellchecker(["YellOw"], ["yeellow", "yllw"]) == ["", ""]

wordlist = ['ee', 'aAk'] | queries = ['Ue', 'K']
sai -> ['ee', 'aAk'] | đúng -> ['ee', '']
ví dụ của đề: ['YellOw', 'YellOw'] (sai) so với ['', '']


Vì sao sai: bỏ nguyên âm làm mất thông tin **vị trí** và **số lượng** nguyên âm. **Đúng:** thay mỗi nguyên âm bằng `*` (giữ nguyên độ dài).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "khớp theo nhiều mức ưu tiên, mỗi mức là một cách coi hai chuỗi là giống nhau" ⇒ mỗi mức một **khóa chuẩn hóa** và một bảng băm; kiểm tra từ mức chặt nhất đến mức lỏng nhất.
- **"Đầu tiên"** ⇒ `setdefault`, hoặc duyệt ngược rồi gán thường.
- Chuẩn hóa chuỗi: `lower()`, thay lớp ký tự bằng ký tự đại diện, sắp xếp ký tự (anagram), v.v.
- Chuẩn hóa `wordlist` **một lần**, tra cứu từng truy vấn `O(L)`.
- Cùng họ: Group Anagrams (49), Find and Replace Pattern (890), Unique Morse Code Words (804), Word Pattern (290).
- Test nên có: từ trùng lặp, mọi nguyên âm hoa, truy vấn khác độ dài, mức 2 đứng sau mức 3, và từ không có nguyên âm.

## 15. Biến thể

1. **Thêm mức "thiếu một chữ cái"**: sinh khóa bằng cách xóa từng ký tự.
2. **Đếm số truy vấn rơi vào từng mức** (bài tập 5).
3. **Nguyên âm gồm cả `y`** hoặc bảng nguyên âm tùy biến: chỉ đổi tập `VOWELS`.
4. Bài liên quan: 49, 890, 804.

Dưới đây là biến thể 1 ở mức đơn giản: đáp án là từ đầu tiên có thể thu được từ truy vấn bằng cách **xóa đúng một ký tự** (không phân biệt hoa/thường), kiểm bằng oracle quét tuyến tính.

In [27]:
def delete_one_fast(wordlist, queries):
    table = {}
    for w in reversed(wordlist):
        low = w.lower()
        table[low] = w
    out = []
    for q in queries:
        low = q.lower()
        ans = ""
        best = None
        first = {}
        for i, w in enumerate(wordlist):
            first.setdefault(w.lower(), i)
        for k in range(len(low)):
            cand = low[:k] + low[k + 1:]
            if cand and cand in first and (best is None or first[cand] < best):
                best = first[cand]
        out.append(wordlist[best] if best is not None else "")
    return out

def delete_one_brute(wordlist, queries):
    out = []
    for q in queries:
        low = q.lower()
        ans = ""
        for w in wordlist:
            if any(w.lower() == low[:k] + low[k + 1:] for k in range(len(low))):
                ans = w
                break
        out.append(ans)
    return out

rng = random.Random(15)
for _ in range(200):
    wl, qs = make_case(rng, rng.randint(1, 10), rng.randint(1, 10), "abAB", 1, 4)
    assert delete_one_fast(wl, qs) == delete_one_brute(wl, qs)
print(delete_one_fast(["Ab", "a", "B"], ["aab", "AB", "bb"]))

['Ab', 'a', 'B']


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét tuyến tính | Theo định nghĩa từng ký tự | `O(Q * W * L)` | `O(1)` | Oracle, ý tưởng ban đầu |
| 2. Ba bảng `setdefault` | Khóa chuẩn hóa mỗi mức | `O((W + Q) * L)` | `O(W * L)` | **Cách nộp chính** |
| 3. Duyệt ngược + một bảng | Từ đầu ghi sau cùng | `O((W + Q) * L)` | `O(W * L)` | Gọn, tránh `setdefault` |
| 4. Liệt kê biến thể | Sinh `5^v` biến thể của truy vấn | `O(Q * 5^v * L)` | `O(W * L)` | Bám sát đề, chỉ để so sánh |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(devowel("HeLLo"), devowel("AEIOU"), devowel("sky"))
print(Solution().spellchecker(["Hello", "hallo"], ["hello", "HELLO", "hullo", "hell"]))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # dòng 1: ba chuỗi
my_guess_2 = None      # dòng 2: danh sách kết quả

### Bài 2 — Tính tay (dễ-vừa)

`wordlist = ["Tea", "tEE", "Toe"]`, `queries = ["TEA", "toa", "Tee", "tie"]`. Ghi mảng kết quả (tính tay theo ba mức ưu tiên).

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm dựng ba bảng và tra cứu.

In [30]:
def exercise_3(wordlist: List[str], queries: List[str]) -> List[str]:
    def mask(s: str) -> str:
        # TODO 1: chữ thường, thay mỗi nguyên âm bằng '*'
        pass

    exact = set(wordlist)
    cap, vow = {}, {}
    for w in wordlist:
        # TODO 2: ghi vào cap và vow nhưng giữ từ ĐẦU TIÊN
        pass
    ans = []
    for q in queries:
        # TODO 3: mức 1 -> mức 2 -> mức 3 -> ""
        pass
    return ans


# for wl, qs, expected in examples:
#     assert exercise_3(wl, qs) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, tự viết `exercise_4(wordlist, queries)` bằng **duyệt ngược** và một `dict` duy nhất với khóa có tiền tố mức.

In [31]:
def exercise_4(wordlist: List[str], queries: List[str]) -> List[str]:
    # TODO
    pass


# for wl, qs, expected in examples:
#     assert exercise_4(wl, qs) == expected

### Bài 5 — Mini challenge: mức nào quyết định? (khó)

Viết `exercise_5(wordlist, queries)` trả về mảng số nguyên: với mỗi truy vấn, `1` nếu khớp chính xác, `2` nếu khớp hoa/thường, `3` nếu khớp nguyên âm, `0` nếu không có đáp án.

```python
exercise_5(["KiTe", "kite", "hare", "Hare"], ["kite", "Kite", "HARE", "Hear", "keti"])  -> [1, 2, 2, 0, 3]
exercise_5(["yellow"], ["YellOw", "yellow", "yollow", "yllw"])                           -> [2, 1, 3, 0]
```
Gợi ý: dùng cùng ba bảng, nhưng ghi lại mức thay vì từ.

In [32]:
def exercise_5(wordlist: List[str], queries: List[str]) -> List[int]:
    # TODO
    pass


# assert exercise_5(["KiTe", "kite", "hare", "Hare"], ["kite", "Kite", "HARE", "Hear", "keti"]) == [1, 2, 2, 0, 3]
# assert exercise_5(["yellow"], ["YellOw", "yellow", "yollow", "yllw"]) == [2, 1, 3, 0]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `devowel` hạ chữ thường rồi thay nguyên âm bằng `*`.
- Mức 2: `"HeLLo"` -> `"hello"` -> `h*ll*`; `"sky"` không có nguyên âm (`y` không tính).
- Mức 3: dòng 1 là `h*ll* ***** sky`. Dòng 2: `"hello"` không có chính xác trong `wordlist` (chỉ có `"Hello"`), nên rơi vào mức 2 và trả `Hello`; `HELLO` cũng mức 2; `hullo` mức 3 (khóa `h*ll*`, từ đầu tiên là `Hello`); `hell` khác độ dài nên `""`.

**Bài 2**
- Mức 1: `TEA` mức 2 (`tea` -> `Tea`).
- Mức 2: `toa` mức 3: khóa `t**`; trong `wordlist` khóa `t**` đầu tiên là `Tea`.
- Mức 3: `Tee` chính xác? Không có `Tee` (chỉ `tEE`): mức 2 -> `tEE`; `tie` -> khóa `t**` -> `Tea`.

**Bài 3**
- Mức 1: `"".join("*" if c in "aeiou" else c for c in s.lower())`.
- Mức 2: `cap.setdefault(w.lower(), w)` và `vow.setdefault(mask(w), w)`.
- Mức 3: `if q in exact ... elif q.lower() in cap ... else vow.get(mask(q), "")`.

**Bài 4**
- Mức 1: `for w in reversed(wordlist)`.
- Mức 2: `table[(2, w.lower())] = w`, `table[(3, mask(w))] = w`.
- Mức 3: tra `(2, ...)` rồi `(3, ...)`.

**Bài 5**
- Mức 1: giống Bài 3 nhưng `append` mã `1/2/3/0`.
- Mức 2: chỉ cần kiểm tra khóa có trong bảng tương ứng.
- Mức 3: dùng `set` cho `cap` và `vow` cũng đủ.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print(devowel("HeLLo"), devowel("AEIOU"), devowel("sky"))
print(Solution().spellchecker(["Hello", "hallo"], ["hello", "HELLO", "hullo", "hell"]))

assert (devowel("HeLLo"), devowel("AEIOU"), devowel("sky")) == ("h*ll*", "*****", "sky")
assert Solution().spellchecker(["Hello", "hallo"], ["hello", "HELLO", "hullo", "hell"]) == ["Hello", "Hello", "Hello", ""]

h*ll* ***** sky
['Hello', 'Hello', 'Hello', '']


### Đáp án Bài 2

`TEA` -> mức 2: `Tea`. `toa` -> không chính xác, mức 2 `toa` không có, mức 3 khóa `t**`: từ đầu tiên là `Tea`. `Tee` -> mức 2 `tee` khớp `tEE` (từ đầu tiên có chuỗi thường `tee`). `tie` -> mức 3 `t**` -> `Tea`.

In [34]:
wl, qs = ["Tea", "tEE", "Toe"], ["TEA", "toa", "Tee", "tie"]
hand_answer_solution = ["Tea", "Tea", "tEE", "Tea"]
assert Solution().spellchecker(wl, qs) == hand_answer_solution == spell_brute(wl, qs)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(wordlist: List[str], queries: List[str]) -> List[str]:
    def mask(s: str) -> str:
        return "".join("*" if c in "aeiou" else c for c in s.lower())

    exact = set(wordlist)
    cap, vow = {}, {}
    for w in wordlist:
        cap.setdefault(w.lower(), w)
        vow.setdefault(mask(w), w)
    ans = []
    for q in queries:
        if q in exact:
            ans.append(q)
        elif q.lower() in cap:
            ans.append(cap[q.lower()])
        else:
            ans.append(vow.get(mask(q), ""))
    return ans

for wl, qs, expected in examples:
    assert exercise_3_solution(wl, qs) == expected
rng = random.Random(3)
for _ in range(200):
    wl, qs = make_case(rng, rng.randint(1, 10), rng.randint(1, 10), "aeAEkK", 1, 4)
    assert exercise_3_solution(wl, qs) == spell_brute(wl, qs)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(wordlist: List[str], queries: List[str]) -> List[str]:
    def mask(s: str) -> str:
        return "".join("*" if c in "aeiou" else c for c in s.lower())

    exact = set(wordlist)
    table = {}
    for w in reversed(wordlist):
        table[(2, w.lower())] = w
        table[(3, mask(w))] = w
    ans = []
    for q in queries:
        if q in exact:
            ans.append(q)
        elif (2, q.lower()) in table:
            ans.append(table[(2, q.lower())])
        else:
            ans.append(table.get((3, mask(q)), ""))
    return ans

for wl, qs, expected in examples:
    assert exercise_4_solution(wl, qs) == expected
rng = random.Random(4)
for _ in range(200):
    wl, qs = make_case(rng, rng.randint(1, 10), rng.randint(1, 10), "aeAEkK", 1, 4)
    assert exercise_4_solution(wl, qs) == spell_brute(wl, qs)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(wordlist: List[str], queries: List[str]) -> List[int]:
    def mask(s: str) -> str:
        return "".join("*" if c in "aeiou" else c for c in s.lower())

    exact = set(wordlist)
    cap = {w.lower() for w in wordlist}
    vow = {mask(w) for w in wordlist}
    out = []
    for q in queries:
        if q in exact:
            out.append(1)
        elif q.lower() in cap:
            out.append(2)
        elif mask(q) in vow:
            out.append(3)
        else:
            out.append(0)
    return out

assert exercise_5_solution(["KiTe", "kite", "hare", "Hare"], ["kite", "Kite", "HARE", "Hear", "keti"]) == [1, 2, 2, 0, 3]
assert exercise_5_solution(["yellow"], ["YellOw", "yellow", "yollow", "yllw"]) == [2, 1, 3, 0]
rng = random.Random(5)
for _ in range(200):
    wl, qs = make_case(rng, rng.randint(1, 10), rng.randint(1, 10), "aeAEkK", 1, 4)
    ref = spell_brute(wl, qs)
    exp = [0 if r == "" else 1 if r == q else 2 if r.lower() == q.lower() else 3 for q, r in zip(qs, ref)]
    assert exercise_5_solution(wl, qs) == exp
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (ba bảng chuẩn hóa)

```python
class Solution:
    def spellchecker(self, wordlist: List[str], queries: List[str]) -> List[str]:
        def mask(s: str) -> str:                               # chuẩn hóa mức 3
            return "".join("*" if c in "aeiou" else c for c in s.lower())

        exact = set(wordlist)                                  # mức 1
        cap, vow = {}, {}
        for w in wordlist:
            cap.setdefault(w.lower(), w)                       # mức 2: từ ĐẦU TIÊN
            vow.setdefault(mask(w), w)                         # mức 3: từ ĐẦU TIÊN

        ans = []
        for q in queries:
            if q in exact:                                     # chính xác > hoa/thường > nguyên âm
                ans.append(q)
            elif q.lower() in cap:
                ans.append(cap[q.lower()])
            else:
                ans.append(vow.get(mask(q), ""))
        return ans
```

Công thức nhớ nhanh: mỗi mức ưu tiên = một khóa chuẩn hóa; gặp "đầu tiên" thì dùng `setdefault`.

### Checklist trước khi submit

- [ ] Kiểm tra khớp **chính xác** (phân biệt hoa/thường) trước tiên và trả `q`?
- [ ] Thứ tự ưu tiên: chính xác, rồi hoa/thường, rồi nguyên âm?
- [ ] `setdefault` (hoặc duyệt ngược) để giữ từ **đầu tiên**, không ghi đè?
- [ ] `.lower()` **trước** khi che nguyên âm?
- [ ] Che bằng `*` (giữ độ dài), không xóa nguyên âm?
- [ ] Trả `""` khi không khớp, và trả từ **như trong `wordlist`** (đúng hoa/thường)?
- [ ] Đã thử: ví dụ 1-2, từ trùng lặp, từ không có nguyên âm, truy vấn khác độ dài, `W = Q = 5000`?